# Consensus Mean Reversion and Momentum

Combines mean-reversion rules and a momentum filter in a walk-forward backtest.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
================================================================================
THRESHOLD REGIME TRADING STRATEGY - VERSION 2.5.2 (MULTI-TIMEFRAME Z-SCORE)
================================================================================

ENHANCEMENTS FROM V2.5.1:
-------------------------
1. Multi-Timeframe Z-Score Consensus:
   - Calculate Z-scores at 3 different lookback periods (short/medium/long)
   - Require alignment across timeframes for stronger signals
   - Reduces false signals from single-timeframe noise

2. Dynamic Percentile-Based Thresholds:
   - Z-score extremes defined by historical percentile rank
   - Adapts to changing volatility environments
   - More robust than fixed -2/+2 thresholds

3. Enhanced Consensus Voting:
   - Multi-TF Z-score gets stronger weight in consensus
   - Can require all timeframes to agree (strict mode)
   - Or majority agreement (moderate mode)

TEMPORAL FLOW (NO LOOK-AHEAD):
------------------------------
Close[T]:   Calculate all indicators (IBS, multi-TF Z-scores, RSI) → signal[T]
Open[T+1]:  Execute signal[T] (signals.shift(1) in backtest)
Close[T+1]: Mark-to-market, calculate P&L

================================================================================
"""

from _market_data import download_price_records
import numpy as np
import pandas as pd
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass, field
from datetime import datetime, timedelta
import warnings
from scipy import stats
from itertools import product
import json

warnings.filterwarnings("ignore")
TICKER = "GOOGL"


@dataclass
class StrategyConfig:
    """Master configuration for Threshold Regime Strategy V2.5.2."""

    ticker: str = TICKER
    benchmark_ticker: str = "URTH"
    start_date: str = "2019-01-01"
    end_date: str = "2025-11-27"
    warmup_buffer_days: int = 700
    vol_lookback: int = 20
    quantile_window: int = 252
    n_regimes: int = 2
    regime_quantile: float = 0.5
    yz_k_numerator: float = 0.34
    yz_k_denominator_constant: float = 1.34
    yz_annualization_factor: int = 252
    persistence_days: int = 3
    use_persistence_filter: bool = True
    zscore_short_lookback: int = 10
    zscore_medium_lookback: int = 20
    zscore_long_lookback: int = 40
    zscore_short_oversold: float = -2.0
    zscore_short_overbought: float = 2.0
    zscore_medium_oversold: float = -1.5
    zscore_medium_overbought: float = 1.5
    zscore_long_oversold: float = -1.0
    zscore_long_overbought: float = 1.0
    use_percentile_thresholds: bool = True
    percentile_lookback: int = 252
    extreme_oversold_percentile: float = 0.05
    extreme_overbought_percentile: float = 0.95
    multitf_consensus_mode: str = "majority"
    multitf_zscore_weight: float = 2.0
    ibs_smoothing: int = 0
    ibs_oversold: float = 0.2
    ibs_overbought: float = 0.8
    zscore_lookback: int = 20
    zscore_oversold: float = -2.0
    zscore_overbought: float = 2.0
    rsi_period: int = 14
    rsi_oversold: float = 30
    rsi_overbought: float = 70
    consensus_mode: str = "moderate"
    min_votes_required: int = 2
    ibs_weight: float = 1.0
    zscore_weight: float = 1.0
    rsi_weight: float = 1.0
    weighted_threshold: float = 2.0
    adaptive_high_vol_votes: int = 3
    adaptive_low_vol_votes: int = 2
    use_consensus_exit: bool = True
    exit_mode: str = "any"
    exit_ibs_long: float = 0.85
    exit_ibs_short: float = 0.15
    exit_zscore_long: float = 1.5
    exit_zscore_short: float = -1.5
    exit_rsi_long: float = 65
    exit_rsi_short: float = 35
    max_wait_days: int = 5
    initial_capital: float = 100000.0
    compounding_mode: bool = True
    leverage: float = 1.0
    low_vol_allocation: float = 1.0
    high_vol_allocation: float = -1.0
    commission_per_share: float = 0.0035
    commission_min: float = 0.35
    commission_max_pct: float = 0.01
    spread_bps: float = 2.0
    market_impact_bps: float = 3.0
    margin_rate_annual: float = 0.06
    short_borrow_fee_annual: float = 0.005
    min_train_window: int = 504
    validation_window: int = 126
    step_size: int = 63
    continuous_equity: bool = True
    vol_lookback_grid: List[int] = field(default_factory=lambda: [10, 15, 20, 30, 42, 63])
    quantile_window_grid: List[int] = field(default_factory=lambda: [126, 189, 252, 378, 504])
    regime_quantile_grid: List[float] = field(default_factory=lambda: [0.3, 0.4, 0.5, 0.6, 0.7])
    persistence_days_grid: List[int] = field(default_factory=lambda: [1, 2, 3, 4, 5])
    consensus_mode_grid: List[str] = field(default_factory=lambda: ["moderate", "strict"])
    min_votes_grid: List[int] = field(default_factory=lambda: [2, 3])
    zscore_short_lookback_grid: List[int] = field(default_factory=lambda: [5, 10, 15])
    zscore_medium_lookback_grid: List[int] = field(default_factory=lambda: [15, 20, 30])
    zscore_long_lookback_grid: List[int] = field(default_factory=lambda: [30, 40, 60])
    multitf_consensus_mode_grid: List[str] = field(default_factory=lambda: ["majority", "strict"])
    rsi_period_grid: List[int] = field(default_factory=lambda: [9, 14, 21])
    rsi_oversold_grid: List[float] = field(default_factory=lambda: [25, 30, 35])
    rsi_overbought_grid: List[float] = field(default_factory=lambda: [65, 70, 75])
    exit_mode_grid: List[str] = field(default_factory=lambda: ["any", "majority"])
    fast_mode: bool = False
    optimization_objective: str = "sharpe"
    max_param_combinations: int = 200
    gri_threshold: float = 0.65
    min_psr: float = 0.95
    min_wfe: float = 0.5
    num_trials: int = 100
    psr_benchmark_sharpe: float = 0.0


class YahooDataFetcher:
    """Yahoo Finance data fetcher."""

    def __init__(self):
        pass

    def fetch_historical_data(self, ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
        print(f"[DATA] Fetching {ticker} from {start_date} to {end_date}...")
        try:
            response_prices = {"historical": download_price_records(ticker, start_date, end_date)}
            data = response_prices
        except RuntimeError as e:
            raise ValueError(f"API request failed for {ticker}: {str(e)}")
        if "historical" not in data or len(data["historical"]) == 0:
            raise ValueError(f"No historical data for {ticker}")
        df = pd.DataFrame(data["historical"])
        df["date"] = pd.to_datetime(df["date"])
        df.set_index("date", inplace=True)
        df.sort_index(inplace=True)
        col_map = {
            "open": "Open",
            "high": "High",
            "low": "Low",
            "close": "Close",
            "adjClose": "Adj Close",
            "volume": "Volume",
        }
        available = {k: v for k, v in col_map.items() if k in df.columns}
        df = df[list(available.keys())].rename(columns=available)
        if "Adj Close" in df.columns and "Close" in df.columns:
            adj_ratio = df["Adj Close"] / df["Close"]
            df["Open"] = df["Open"] * adj_ratio
            df["High"] = df["High"] * adj_ratio
            df["Low"] = df["Low"] * adj_ratio
            df["Close"] = df["Adj Close"]
        print(f"[DATA] Retrieved {len(df)} trading days for {ticker}")
        return df


class YangZhangVolatility:

    @staticmethod
    def calculate(df: pd.DataFrame, window: int, config: StrategyConfig) -> pd.Series:
        """
        Yang-Zhang volatility - available at Close[T] using data up to T.
        NO SHIFT needed - naturally aligned.
        """
        log_open = np.log(df["Open"])
        log_high = np.log(df["High"])
        log_low = np.log(df["Low"])
        log_close = np.log(df["Close"])
        log_close_prev = log_close.shift(1)
        overnight_ret = log_open - log_close_prev
        var_overnight = overnight_ret.rolling(window).var()
        open_close_ret = log_close - log_open
        var_open_close = open_close_ret.rolling(window).var()
        rs_component = (log_high - log_close) * (log_high - log_open) + (log_low - log_close) * (
            log_low - log_open
        )
        var_rs = rs_component.rolling(window).mean()
        k = config.yz_k_numerator / (config.yz_k_denominator_constant + (window + 1) / (window - 1))
        yz_variance = var_overnight + k * var_open_close + (1 - k) * var_rs
        return np.sqrt(yz_variance) * np.sqrt(config.yz_annualization_factor)


class MultiTimeframeZScore:
    """
    Multi-timeframe Z-score analysis for mean reversion signals.

    METHODOLOGY:
    -----------
    1. Calculate Z-scores at multiple lookback periods (short/medium/long)
    2. Apply fixed thresholds OR percentile-based dynamic thresholds
    3. Require consensus across timeframes for signal generation
    4. Reduces false signals from single-timeframe noise

    TEMPORAL GUARANTEE:
    ------------------
    All indicators use data up to time T, available at Close[T].
    NO SHIFT applied - execution lag handled in backtest engine.
    """

    def __init__(self, config: StrategyConfig):
        self.config = config

    def calculate_single_zscore(
        self, df: pd.DataFrame, lookback: int, use_percentile: bool = False
    ) -> Tuple[pd.Series, pd.Series, pd.Series]:
        """
        Calculate Z-score for a single lookback period.

        Args:
            df: DataFrame with Close prices
            lookback: Rolling window size
            use_percentile: If True, use percentile-based thresholds

        Returns:
            Tuple of (zscore, oversold_signal, overbought_signal)
        """
        sma = df["Close"].rolling(lookback).mean()
        std = df["Close"].rolling(lookback).std()
        zscore = np.where(std > 0, (df["Close"] - sma) / std, 0)
        zscore_series = pd.Series(zscore, index=df.index, name=f"zscore_{lookback}")
        if use_percentile:
            percentile_rank = zscore_series.rolling(self.config.percentile_lookback).rank(pct=True)
            oversold = percentile_rank <= self.config.extreme_oversold_percentile
            overbought = percentile_rank >= self.config.extreme_overbought_percentile
        elif lookback == self.config.zscore_short_lookback:
            oversold = zscore_series < self.config.zscore_short_oversold
            overbought = zscore_series > self.config.zscore_short_overbought
        elif lookback == self.config.zscore_medium_lookback:
            oversold = zscore_series < self.config.zscore_medium_oversold
            overbought = zscore_series > self.config.zscore_medium_overbought
        else:
            oversold = zscore_series < self.config.zscore_long_oversold
            overbought = zscore_series > self.config.zscore_long_overbought
        return (zscore_series, oversold.astype(int), overbought.astype(int))

    def calculate_multi_timeframe(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Calculate Z-scores across multiple timeframes and generate consensus.

        CRITICAL: NO .shift(1) - all data available at Close[T]

        Returns:
            DataFrame with:
            - zscore_short, zscore_medium, zscore_long: Raw Z-scores
            - zscore_*_oversold/overbought: Individual signals
            - multitf_zscore_long/short: Consensus signals
            - multitf_zscore_strength: Number of agreeing timeframes
        """
        results = pd.DataFrame(index=df.index)
        lookbacks = [
            self.config.zscore_short_lookback,
            self.config.zscore_medium_lookback,
            self.config.zscore_long_lookback,
        ]
        oversold_votes = []
        overbought_votes = []
        for lookback in lookbacks:
            zscore, oversold, overbought = self.calculate_single_zscore(
                df, lookback, use_percentile=self.config.use_percentile_thresholds
            )
            if lookback == self.config.zscore_short_lookback:
                results["zscore_short"] = zscore
                results["zscore_short_oversold"] = oversold
                results["zscore_short_overbought"] = overbought
            elif lookback == self.config.zscore_medium_lookback:
                results["zscore_medium"] = zscore
                results["zscore_medium_oversold"] = oversold
                results["zscore_medium_overbought"] = overbought
            else:
                results["zscore_long"] = zscore
                results["zscore_long_oversold"] = oversold
                results["zscore_long_overbought"] = overbought
            oversold_votes.append(oversold)
            overbought_votes.append(overbought)
        total_oversold_votes = sum(oversold_votes)
        total_overbought_votes = sum(overbought_votes)
        if self.config.multitf_consensus_mode == "strict":
            results["multitf_zscore_long"] = total_oversold_votes == 3
            results["multitf_zscore_short"] = total_overbought_votes == 3
            results["multitf_zscore_strength"] = np.maximum(
                total_oversold_votes, total_overbought_votes
            )
        elif self.config.multitf_consensus_mode == "majority":
            results["multitf_zscore_long"] = total_oversold_votes >= 2
            results["multitf_zscore_short"] = total_overbought_votes >= 2
            results["multitf_zscore_strength"] = np.maximum(
                total_oversold_votes, total_overbought_votes
            )
        else:
            raise ValueError(
                f"Unknown multitf_consensus_mode: {self.config.multitf_consensus_mode}"
            )
        results["multitf_oversold_votes"] = total_oversold_votes
        results["multitf_overbought_votes"] = total_overbought_votes
        return results


class ConsensusTimingIndicator:
    """
    Multi-indicator consensus system with enhanced multi-timeframe Z-score.

    ENHANCEMENTS IN V2.5.2:
    - Integrated multi-timeframe Z-score analysis
    - Multi-TF consensus gets higher weight in voting
    - Legacy single-TF Z-score maintained for backward compatibility
    """

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.multitf_zscore = MultiTimeframeZScore(config)

    def calculate_ibs(self, df: pd.DataFrame) -> pd.Series:
        """IBS: (Close - Low) / (High - Low) - NO SHIFT"""
        high_low_range = df["High"] - df["Low"]
        ibs = np.where(high_low_range > 0, (df["Close"] - df["Low"]) / high_low_range, 0.5)
        ibs_series = pd.Series(ibs, index=df.index, name="IBS")
        if self.config.ibs_smoothing > 0:
            ibs_series = ibs_series.rolling(self.config.ibs_smoothing).mean()
        return ibs_series

    def calculate_zscore_legacy(self, df: pd.DataFrame) -> pd.Series:
        """
        Legacy single-timeframe Z-Score for backward compatibility.

        NOTE: V2.5.2 uses multi-timeframe Z-score by default.
        This is kept for comparison purposes.
        """
        sma = df["Close"].rolling(self.config.zscore_lookback).mean()
        std = df["Close"].rolling(self.config.zscore_lookback).std()
        zscore = np.where(std > 0, (df["Close"] - sma) / std, 0)
        return pd.Series(zscore, index=df.index, name="Z_Score_Legacy")

    def calculate_rsi(self, df: pd.DataFrame) -> pd.Series:
        """RSI: 100 - (100 / (1 + RS)) - NO SHIFT"""
        delta = df["Close"].diff()
        gain = delta.where(delta > 0, 0)
        loss = -delta.where(delta < 0, 0)
        avg_gain = gain.ewm(
            alpha=1 / self.config.rsi_period, min_periods=self.config.rsi_period, adjust=False
        ).mean()
        avg_loss = loss.ewm(
            alpha=1 / self.config.rsi_period, min_periods=self.config.rsi_period, adjust=False
        ).mean()
        rs = np.where(avg_loss > 0, avg_gain / avg_loss, 100)
        rsi = 100 - 100 / (1 + rs)
        return pd.Series(rsi, index=df.index, name="RSI")

    def generate_consensus_signals(
        self, df: pd.DataFrame, vol_series: Optional[pd.Series] = None
    ) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """
        Generate consensus signals with enhanced multi-TF Z-score.

        CRITICAL ENHANCEMENT:
        - Multi-TF Z-score consensus replaces single-TF Z-score in voting
        - Can receive higher weight via multitf_zscore_weight parameter
        - All indicators available at Close[T], no shift
        """
        df = df.copy()
        df["ibs"] = self.calculate_ibs(df)
        df["rsi"] = self.calculate_rsi(df)
        multitf_results = self.multitf_zscore.calculate_multi_timeframe(df)
        df = pd.concat([df, multitf_results], axis=1)
        df["zscore_legacy"] = self.calculate_zscore_legacy(df)
        df["ibs_vote_long"] = (df["ibs"] < self.config.ibs_oversold).astype(int)
        df["ibs_vote_short"] = (df["ibs"] > self.config.ibs_overbought).astype(int)
        df["rsi_vote_long"] = (df["rsi"] < self.config.rsi_oversold).astype(int)
        df["rsi_vote_short"] = (df["rsi"] > self.config.rsi_overbought).astype(int)
        df["multitf_zscore_vote_long"] = df["multitf_zscore_long"].astype(int)
        df["multitf_zscore_vote_short"] = df["multitf_zscore_short"].astype(int)
        df["zscore_vote_long_legacy"] = (df["zscore_legacy"] < self.config.zscore_oversold).astype(
            int
        )
        df["zscore_vote_short_legacy"] = (
            df["zscore_legacy"] > self.config.zscore_overbought
        ).astype(int)
        if self.config.consensus_mode == "strict":
            df["consensus_long"] = (
                (df["ibs_vote_long"] == 1)
                & (df["multitf_zscore_vote_long"] == 1)
                & (df["rsi_vote_long"] == 1)
            )
            df["consensus_short"] = (
                (df["ibs_vote_short"] == 1)
                & (df["multitf_zscore_vote_short"] == 1)
                & (df["rsi_vote_short"] == 1)
            )
            df["consensus_strength"] = np.where(
                df["consensus_long"] | df["consensus_short"],
                3,
                df["ibs_vote_long"] + df["multitf_zscore_vote_long"] + df["rsi_vote_long"],
            )
        elif self.config.consensus_mode == "moderate":
            long_votes = df["ibs_vote_long"] + df["multitf_zscore_vote_long"] + df["rsi_vote_long"]
            short_votes = (
                df["ibs_vote_short"] + df["multitf_zscore_vote_short"] + df["rsi_vote_short"]
            )
            df["consensus_long"] = long_votes >= self.config.min_votes_required
            df["consensus_short"] = short_votes >= self.config.min_votes_required
            df["consensus_strength"] = np.maximum(long_votes, short_votes)
        elif self.config.consensus_mode == "weighted":
            long_score = (
                df["ibs_vote_long"] * self.config.ibs_weight
                + df["multitf_zscore_vote_long"] * self.config.multitf_zscore_weight
                + df["rsi_vote_long"] * self.config.rsi_weight
            )
            short_score = (
                df["ibs_vote_short"] * self.config.ibs_weight
                + df["multitf_zscore_vote_short"] * self.config.multitf_zscore_weight
                + df["rsi_vote_short"] * self.config.rsi_weight
            )
            df["consensus_long"] = long_score >= self.config.weighted_threshold
            df["consensus_short"] = short_score >= self.config.weighted_threshold
            df["consensus_strength"] = np.maximum(long_score, short_score)
        elif self.config.consensus_mode == "adaptive":
            if vol_series is None:
                raise ValueError("Adaptive mode requires vol_series")
            vol_threshold = vol_series.rolling(252).quantile(0.5).shift(1)
            high_vol_regime = vol_series.shift(1) > vol_threshold
            long_votes = df["ibs_vote_long"] + df["multitf_zscore_vote_long"] + df["rsi_vote_long"]
            short_votes = (
                df["ibs_vote_short"] + df["multitf_zscore_vote_short"] + df["rsi_vote_short"]
            )
            votes_required = np.where(
                high_vol_regime,
                self.config.adaptive_high_vol_votes,
                self.config.adaptive_low_vol_votes,
            )
            df["consensus_long"] = long_votes >= votes_required
            df["consensus_short"] = short_votes >= votes_required
            df["consensus_strength"] = np.maximum(long_votes, short_votes)
            df["votes_required"] = votes_required
            df["high_vol_regime"] = high_vol_regime
        else:
            raise ValueError(f"Unknown consensus mode: {self.config.consensus_mode}")
        if self.config.use_consensus_exit:
            df["multitf_exit_long"] = (
                (df["zscore_short"] > self.config.exit_zscore_long)
                | (df["zscore_medium"] > self.config.exit_zscore_long * 0.75)
            ).astype(int)
            df["multitf_exit_short"] = (
                (df["zscore_short"] < self.config.exit_zscore_short)
                | (df["zscore_medium"] < self.config.exit_zscore_short * 0.75)
            ).astype(int)
            df["ibs_exit_long"] = (df["ibs"] > self.config.exit_ibs_long).astype(int)
            df["ibs_exit_short"] = (df["ibs"] < self.config.exit_ibs_short).astype(int)
            df["rsi_exit_long"] = (df["rsi"] > self.config.exit_rsi_long).astype(int)
            df["rsi_exit_short"] = (df["rsi"] < self.config.exit_rsi_short).astype(int)
            if self.config.exit_mode == "any":
                df["exit_long"] = (
                    (df["ibs_exit_long"] == 1)
                    | (df["multitf_exit_long"] == 1)
                    | (df["rsi_exit_long"] == 1)
                )
                df["exit_short"] = (
                    (df["ibs_exit_short"] == 1)
                    | (df["multitf_exit_short"] == 1)
                    | (df["rsi_exit_short"] == 1)
                )
            elif self.config.exit_mode == "majority":
                exit_long_votes = (
                    df["ibs_exit_long"] + df["multitf_exit_long"] + df["rsi_exit_long"]
                )
                exit_short_votes = (
                    df["ibs_exit_short"] + df["multitf_exit_short"] + df["rsi_exit_short"]
                )
                df["exit_long"] = exit_long_votes >= 2
                df["exit_short"] = exit_short_votes >= 2
            elif self.config.exit_mode == "all":
                df["exit_long"] = (
                    (df["ibs_exit_long"] == 1)
                    & (df["multitf_exit_long"] == 1)
                    & (df["rsi_exit_long"] == 1)
                )
                df["exit_short"] = (
                    (df["ibs_exit_short"] == 1)
                    & (df["multitf_exit_short"] == 1)
                    & (df["rsi_exit_short"] == 1)
                )
        else:
            df["exit_long"] = False
            df["exit_short"] = False
        signals_df = pd.DataFrame(index=df.index)
        signals_df["consensus_long"] = df["consensus_long"]
        signals_df["consensus_short"] = df["consensus_short"]
        signals_df["exit_long"] = df["exit_long"]
        signals_df["exit_short"] = df["exit_short"]
        signals_df["consensus_strength"] = df["consensus_strength"]
        diagnostics_df = pd.DataFrame(index=df.index)
        diagnostics_df["ibs"] = df["ibs"]
        diagnostics_df["rsi"] = df["rsi"]
        diagnostics_df["zscore_legacy"] = df["zscore_legacy"]
        diagnostics_df["zscore_short"] = df["zscore_short"]
        diagnostics_df["zscore_medium"] = df["zscore_medium"]
        diagnostics_df["zscore_long"] = df["zscore_long"]
        diagnostics_df["multitf_zscore_strength"] = df["multitf_zscore_strength"]
        diagnostics_df["multitf_oversold_votes"] = df["multitf_oversold_votes"]
        diagnostics_df["multitf_overbought_votes"] = df["multitf_overbought_votes"]
        diagnostics_df["ibs_vote_long"] = df["ibs_vote_long"]
        diagnostics_df["ibs_vote_short"] = df["ibs_vote_short"]
        diagnostics_df["multitf_zscore_vote_long"] = df["multitf_zscore_vote_long"]
        diagnostics_df["multitf_zscore_vote_short"] = df["multitf_zscore_vote_short"]
        diagnostics_df["rsi_vote_long"] = df["rsi_vote_long"]
        diagnostics_df["rsi_vote_short"] = df["rsi_vote_short"]
        diagnostics_df["consensus_strength"] = df["consensus_strength"]
        if self.config.consensus_mode == "adaptive":
            diagnostics_df["votes_required"] = df["votes_required"]
            diagnostics_df["high_vol_regime"] = df["high_vol_regime"]
        return (signals_df, diagnostics_df)


class ThresholdRegimeDetectorWithConsensus:
    """Regime detection + enhanced consensus timing with multi-TF Z-score."""

    def __init__(
        self,
        config: StrategyConfig,
        vol_lookback: Optional[int] = None,
        quantile_window: Optional[int] = None,
        regime_quantile: Optional[float] = None,
        persistence_days: Optional[int] = None,
        max_wait_days: Optional[int] = None,
        consensus_mode: Optional[str] = None,
        min_votes: Optional[int] = None,
        zscore_short_lookback: Optional[int] = None,
        zscore_medium_lookback: Optional[int] = None,
        zscore_long_lookback: Optional[int] = None,
        multitf_consensus_mode: Optional[str] = None,
        rsi_period: Optional[int] = None,
        rsi_oversold: Optional[float] = None,
        rsi_overbought: Optional[float] = None,
        exit_mode: Optional[str] = None,
    ):
        self.config = config
        self.vol_lookback = vol_lookback or config.vol_lookback
        self.quantile_window = quantile_window or config.quantile_window
        self.regime_quantile = regime_quantile or config.regime_quantile
        self.persistence_days = persistence_days or config.persistence_days
        self.max_wait_days = max_wait_days if max_wait_days is not None else config.max_wait_days
        if consensus_mode is not None:
            self.config.consensus_mode = consensus_mode
        if min_votes is not None:
            self.config.min_votes_required = min_votes
        if zscore_short_lookback is not None:
            self.config.zscore_short_lookback = zscore_short_lookback
        if zscore_medium_lookback is not None:
            self.config.zscore_medium_lookback = zscore_medium_lookback
        if zscore_long_lookback is not None:
            self.config.zscore_long_lookback = zscore_long_lookback
        if multitf_consensus_mode is not None:
            self.config.multitf_consensus_mode = multitf_consensus_mode
        if rsi_period is not None:
            self.config.rsi_period = rsi_period
        if rsi_oversold is not None:
            self.config.rsi_oversold = rsi_oversold
        if rsi_overbought is not None:
            self.config.rsi_overbought = rsi_overbought
        if exit_mode is not None:
            self.config.exit_mode = exit_mode
        self.consensus_indicator = ConsensusTimingIndicator(config)

    def detect_regimes(self, df: pd.DataFrame) -> pd.DataFrame:
        """Detect volatility regimes."""
        df = df.copy()
        df["volatility"] = YangZhangVolatility.calculate(df, self.vol_lookback, self.config)
        df["vol_threshold"] = (
            df["volatility"]
            .rolling(self.quantile_window, closed="left")
            .quantile(self.regime_quantile)
        )
        df["regime_raw"] = (df["volatility"] > df["vol_threshold"]).astype(int)
        if self.config.use_persistence_filter and self.persistence_days > 1:
            df["regime"] = self._apply_persistence_filter_retrospective(
                df["regime_raw"], self.persistence_days
            )
        else:
            df["regime"] = df["regime_raw"]
        df["regime_name"] = df["regime"].map({0: "Low Vol (LONG)", 1: "High Vol (SHORT)"})
        df["returns"] = df["Close"].pct_change()
        return df

    def _apply_persistence_filter_retrospective(
        self, raw_regime: pd.Series, persistence_days: int
    ) -> pd.Series:
        """Retrospective persistence filter."""
        filtered = raw_regime.copy()
        regime_streak = pd.Series(index=raw_regime.index, dtype=int)
        prev_val = None
        streak = 0
        for idx in raw_regime.index:
            val = raw_regime.loc[idx]
            if pd.isna(val):
                regime_streak.loc[idx] = 0
                continue
            if val == prev_val:
                streak += 1
            else:
                streak = 1
                prev_val = val
            regime_streak.loc[idx] = streak
        confirmed_regime = pd.Series(index=raw_regime.index, dtype=float)
        current_confirmed = None
        for idx in raw_regime.index:
            val = raw_regime.loc[idx]
            streak = regime_streak.loc[idx]
            if pd.isna(val):
                confirmed_regime.loc[idx] = np.nan
                continue
            if current_confirmed is None:
                if streak >= persistence_days:
                    current_confirmed = val
                confirmed_regime.loc[idx] = val if current_confirmed is None else current_confirmed
            else:
                if val != current_confirmed and streak >= persistence_days:
                    current_confirmed = val
                confirmed_regime.loc[idx] = current_confirmed
        return confirmed_regime

    def generate_signals_with_consensus(
        self, df: pd.DataFrame
    ) -> Tuple[pd.Series, pd.DataFrame, pd.DataFrame]:
        """
        Generate signals with enhanced multi-TF Z-score consensus.

        CRITICAL: Indicators have NO shift, signals will be shifted in backtest.
        """
        df = self.detect_regimes(df)
        consensus_signals, consensus_diag = self.consensus_indicator.generate_consensus_signals(
            df, vol_series=df["volatility"] if self.config.consensus_mode == "adaptive" else None
        )
        signals = pd.Series(index=df.index, dtype=float)
        signals[:] = np.nan
        diagnostics = pd.DataFrame(index=df.index)
        diagnostics["regime"] = df["regime"]
        diagnostics["volatility"] = df["volatility"]
        diagnostics["vol_threshold"] = df["vol_threshold"]
        diagnostics["state"] = "UNKNOWN"
        diagnostics["signal"] = np.nan
        diagnostics["days_waiting"] = 0
        diagnostics["entry_reason"] = ""
        diagnostics["consensus_strength"] = consensus_signals["consensus_strength"]
        current_state = "FLAT"
        days_waiting = 0
        valid_indices = df.dropna(subset=["regime"]).index
        if len(valid_indices) == 0:
            return (signals, diagnostics, consensus_diag)
        for i, idx in enumerate(valid_indices):
            regime = df.loc[idx, "regime"]
            if i > 0:
                prev_regime = df.loc[valid_indices[i - 1], "regime"]
                regime_changed = regime != prev_regime and (not pd.isna(prev_regime))
            else:
                prev_regime = np.nan
                regime_changed = False
            if regime == 0:
                target_direction = "LONG"
                target_alloc = self.config.low_vol_allocation
            else:
                target_direction = "SHORT"
                target_alloc = self.config.high_vol_allocation
            consensus_long = consensus_signals.loc[idx, "consensus_long"]
            consensus_short = consensus_signals.loc[idx, "consensus_short"]
            exit_long = consensus_signals.loc[idx, "exit_long"]
            exit_short = consensus_signals.loc[idx, "exit_short"]
            entry_reason = ""
            if current_state == "FLAT":
                if target_direction == "LONG":
                    if consensus_long:
                        current_state = "LONG"
                        signals.loc[idx] = target_alloc
                        entry_reason = f"consensus_long_{int(consensus_signals.loc[idx, 'consensus_strength'])}votes"
                        days_waiting = 0
                    else:
                        current_state = "WAITING_LONG"
                        signals.loc[idx] = 0.0
                        days_waiting = 1
                        entry_reason = "waiting_consensus_long"
                elif consensus_short:
                    current_state = "SHORT"
                    signals.loc[idx] = target_alloc
                    entry_reason = f"consensus_short_{int(consensus_signals.loc[idx, 'consensus_strength'])}votes"
                    days_waiting = 0
                else:
                    current_state = "WAITING_SHORT"
                    signals.loc[idx] = 0.0
                    days_waiting = 1
                    entry_reason = "waiting_consensus_short"
            elif current_state == "WAITING_LONG":
                if regime_changed and target_direction == "SHORT":
                    if consensus_short:
                        current_state = "SHORT"
                        signals.loc[idx] = target_alloc
                        entry_reason = "regime_flip_consensus_short"
                    else:
                        current_state = "WAITING_SHORT"
                        signals.loc[idx] = 0.0
                        entry_reason = "regime_flip_waiting_short"
                    days_waiting = 0
                elif consensus_long:
                    current_state = "LONG"
                    signals.loc[idx] = target_alloc
                    entry_reason = f"consensus_long_after_{days_waiting}d"
                    days_waiting = 0
                elif self.max_wait_days > 0 and days_waiting >= self.max_wait_days:
                    current_state = "LONG"
                    signals.loc[idx] = target_alloc
                    entry_reason = f"forced_after_{days_waiting}d"
                    days_waiting = 0
                else:
                    signals.loc[idx] = 0.0
                    days_waiting += 1
                    entry_reason = f"waiting_{days_waiting}d"
            elif current_state == "WAITING_SHORT":
                if regime_changed and target_direction == "LONG":
                    if consensus_long:
                        current_state = "LONG"
                        signals.loc[idx] = target_alloc
                        entry_reason = "regime_flip_consensus_long"
                    else:
                        current_state = "WAITING_LONG"
                        signals.loc[idx] = 0.0
                        entry_reason = "regime_flip_waiting_long"
                    days_waiting = 0
                elif consensus_short:
                    current_state = "SHORT"
                    signals.loc[idx] = target_alloc
                    entry_reason = f"consensus_short_after_{days_waiting}d"
                    days_waiting = 0
                elif self.max_wait_days > 0 and days_waiting >= self.max_wait_days:
                    current_state = "SHORT"
                    signals.loc[idx] = target_alloc
                    entry_reason = f"forced_after_{days_waiting}d"
                    days_waiting = 0
                else:
                    signals.loc[idx] = 0.0
                    days_waiting += 1
                    entry_reason = f"waiting_{days_waiting}d"
            elif current_state == "LONG":
                if regime_changed and target_direction == "SHORT":
                    if consensus_short:
                        current_state = "SHORT"
                        signals.loc[idx] = target_alloc
                        entry_reason = "regime_flip_consensus_short"
                    else:
                        current_state = "WAITING_SHORT"
                        signals.loc[idx] = 0.0
                        days_waiting = 1
                        entry_reason = "regime_flip_waiting_short"
                elif self.config.use_consensus_exit and exit_long:
                    current_state = "FLAT"
                    signals.loc[idx] = 0.0
                    entry_reason = "consensus_exit_long"
                else:
                    signals.loc[idx] = target_alloc
                    entry_reason = "holding_long"
            elif current_state == "SHORT":
                if regime_changed and target_direction == "LONG":
                    if consensus_long:
                        current_state = "LONG"
                        signals.loc[idx] = target_alloc
                        entry_reason = "regime_flip_consensus_long"
                    else:
                        current_state = "WAITING_LONG"
                        signals.loc[idx] = 0.0
                        days_waiting = 1
                        entry_reason = "regime_flip_waiting_long"
                elif self.config.use_consensus_exit and exit_short:
                    current_state = "FLAT"
                    signals.loc[idx] = 0.0
                    entry_reason = "consensus_exit_short"
                else:
                    signals.loc[idx] = target_alloc
                    entry_reason = "holding_short"
            diagnostics.loc[idx, "state"] = current_state
            diagnostics.loc[idx, "signal"] = signals.loc[idx]
            diagnostics.loc[idx, "days_waiting"] = days_waiting
            diagnostics.loc[idx, "entry_reason"] = entry_reason
        return (signals, diagnostics, consensus_diag)


class TransactionCostCalculator:

    def __init__(self, config: StrategyConfig):
        self.config = config

    def calculate_total_cost(self, shares: float, price: float) -> float:
        shares = abs(shares)
        trade_value = shares * price
        raw_comm = shares * self.config.commission_per_share
        commission = max(
            self.config.commission_min, min(raw_comm, trade_value * self.config.commission_max_pct)
        )
        spread = trade_value * (self.config.spread_bps / 10000)
        impact = trade_value * (self.config.market_impact_bps / 10000)
        return commission + spread + impact

    def calculate_margin_interest(self, borrowed_value: float, days: int = 1) -> float:
        daily_rate = self.config.margin_rate_annual / 252
        return abs(borrowed_value) * daily_rate * days

    def calculate_short_borrow_fee(self, short_value: float, days: int = 1) -> float:
        daily_rate = self.config.short_borrow_fee_annual / 252
        return abs(short_value) * daily_rate * days


class BacktestEngine:
    """Backtest engine - maintains signals.shift(1) for execution lag."""

    def __init__(self, config: StrategyConfig, benchmark_returns: Optional[pd.Series] = None):
        self.config = config
        self.cost_calculator = TransactionCostCalculator(config)
        self.benchmark_returns = benchmark_returns

    def run_backtest(self, df: pd.DataFrame, signals: pd.Series) -> pd.DataFrame:
        """Run backtest with SINGLE shift for execution."""
        df = df.copy()
        df["target_allocation"] = signals.shift(1)
        df["close_prev"] = df["Close"].shift(1)
        df["open_prev"] = df["Open"].shift(1)
        results = {
            "date": [],
            "open_price": [],
            "close_price": [],
            "target_alloc": [],
            "shares": [],
            "position_value": [],
            "cash": [],
            "portfolio_value": [],
            "daily_return": [],
            "gross_return": [],
            "transaction_cost": [],
            "margin_interest": [],
            "borrow_fee": [],
            "net_return": [],
            "benchmark_return": [],
            "is_execution_day": [],
            "regime": [],
            "position_type": [],
            "sizing_price": [],
            "pnl_overnight": [],
            "pnl_intraday": [],
        }
        cash = self.config.initial_capital
        shares = 0.0
        executed_signal = None
        valid_dates = df.dropna(subset=["target_allocation", "Open", "Close", "close_prev"]).index
        if len(valid_dates) == 0:
            raise ValueError("No valid trading dates")
        for i, date in enumerate(valid_dates):
            open_price = df.loc[date, "Open"]
            close_price = df.loc[date, "Close"]
            close_prev = df.loc[date, "close_prev"]
            open_prev = df.loc[date, "open_prev"]
            target_alloc = df.loc[date, "target_allocation"]
            regime = df.loc[date, "regime"] if "regime" in df.columns else np.nan
            signal_changed = executed_signal is None or abs(target_alloc - executed_signal) > 0.01
            is_execution_day = signal_changed
            transaction_cost = 0.0
            margin_interest = 0.0
            borrow_fee = 0.0
            gross_return = 0.0
            pnl_overnight = 0.0
            pnl_intraday = 0.0
            if signal_changed:
                if self.config.compounding_mode:
                    position_value_prev = shares * close_prev
                    sizing_capital = cash + position_value_prev
                else:
                    sizing_capital = self.config.initial_capital
                target_value = target_alloc * sizing_capital * self.config.leverage
                new_shares = target_value / close_prev if close_prev > 0 else 0
                shares_change = new_shares - shares
                if abs(shares_change) > 0.001:
                    transaction_cost = self.cost_calculator.calculate_total_cost(
                        shares_change, open_price
                    )
                if shares != 0 and close_prev > 0:
                    overnight_return = open_price / close_prev - 1
                    pnl_overnight = shares * close_prev * overnight_return
                else:
                    pnl_overnight = 0.0
                old_shares = shares
                shares = new_shares
                executed_signal = target_alloc
                if shares != 0 and open_price > 0:
                    intraday_return = close_price / open_price - 1
                    pnl_intraday = shares * open_price * intraday_return
                else:
                    pnl_intraday = 0.0
                gross_return = pnl_overnight + pnl_intraday
                trade_cash_flow = (old_shares - new_shares) * open_price
                cash = cash + trade_cash_flow - transaction_cost + pnl_overnight
            elif shares != 0 and close_prev > 0:
                close_return = close_price / close_prev - 1
                gross_return = shares * close_prev * close_return
            else:
                gross_return = 0.0
            if shares < 0:
                short_value = abs(shares * close_price)
                margin_interest = self.cost_calculator.calculate_margin_interest(short_value)
                borrow_fee = self.cost_calculator.calculate_short_borrow_fee(short_value)
                cash = cash - margin_interest - borrow_fee
            position_value = shares * close_price
            portfolio_value = cash + position_value
            net_return = gross_return - transaction_cost - margin_interest - borrow_fee
            if i > 0 and results["portfolio_value"]:
                prev_pv = results["portfolio_value"][-1]
                daily_return = portfolio_value / prev_pv - 1 if prev_pv > 0 else 0
            else:
                daily_return = 0.0
            if self.benchmark_returns is not None and date in self.benchmark_returns.index:
                benchmark_return = self.benchmark_returns.loc[date]
            else:
                benchmark_return = close_price / close_prev - 1 if close_prev > 0 else 0
            if shares > 0.001:
                position_type = "LONG"
            elif shares < -0.001:
                position_type = "SHORT"
            else:
                position_type = "FLAT"
            results["date"].append(date)
            results["open_price"].append(open_price)
            results["close_price"].append(close_price)
            results["target_alloc"].append(target_alloc)
            results["shares"].append(shares)
            results["position_value"].append(position_value)
            results["cash"].append(cash)
            results["portfolio_value"].append(portfolio_value)
            results["daily_return"].append(daily_return)
            results["gross_return"].append(gross_return)
            results["transaction_cost"].append(transaction_cost)
            results["margin_interest"].append(margin_interest)
            results["borrow_fee"].append(borrow_fee)
            results["net_return"].append(net_return)
            results["benchmark_return"].append(benchmark_return)
            results["is_execution_day"].append(is_execution_day)
            results["regime"].append(regime)
            results["position_type"].append(position_type)
            results["sizing_price"].append(close_prev)
            results["pnl_overnight"].append(pnl_overnight)
            results["pnl_intraday"].append(pnl_intraday)
        results_df = pd.DataFrame(results)
        results_df.set_index("date", inplace=True)
        results_df["cumulative_return"] = (
            results_df["portfolio_value"] / self.config.initial_capital - 1
        )
        results_df["cumulative_benchmark"] = (1 + results_df["benchmark_return"]).cumprod() - 1
        results_df["strategy_return"] = results_df["daily_return"]
        return results_df


class PerformanceAnalytics:

    def __init__(self, config: StrategyConfig):
        self.config = config

    def calculate_basic_metrics(
        self,
        returns: pd.Series,
        benchmark_returns: Optional[pd.Series] = None,
        annualization_factor: int = 252,
    ) -> Dict[str, float]:
        returns = returns.dropna()
        if len(returns) < 20:
            return {
                "error": "Insufficient data",
                "sharpe_ratio": 0,
                "num_observations": len(returns),
            }
        total_return = (1 + returns).prod() - 1
        n_years = len(returns) / annualization_factor
        annual_return = (1 + total_return) ** (1 / n_years) - 1 if n_years > 0 else 0
        volatility = returns.std() * np.sqrt(annualization_factor)
        sharpe = annual_return / volatility if volatility > 0 else 0
        downside = returns[returns < 0]
        downside_dev = downside.std() * np.sqrt(annualization_factor) if len(downside) > 0 else 0
        sortino = annual_return / downside_dev if downside_dev > 0 else 0
        cumulative = (1 + returns).cumprod()
        running_max = cumulative.cummax()
        drawdown = (cumulative - running_max) / running_max
        max_drawdown = drawdown.min()
        calmar = annual_return / abs(max_drawdown) if max_drawdown != 0 else 0
        metrics = {
            "total_return": total_return,
            "annual_return": annual_return,
            "volatility": volatility,
            "sharpe_ratio": sharpe,
            "sortino_ratio": sortino,
            "max_drawdown": max_drawdown,
            "calmar_ratio": calmar,
            "skewness": returns.skew(),
            "kurtosis": returns.kurtosis() + 3,
            "num_observations": len(returns),
        }
        if benchmark_returns is not None:
            benchmark_returns = benchmark_returns.reindex(returns.index).dropna()
            if len(benchmark_returns) > 20:
                benchmark_total = (1 + benchmark_returns).prod() - 1
                active_returns = returns - benchmark_returns.reindex(returns.index).fillna(0)
                tracking_error = active_returns.std() * np.sqrt(annualization_factor)
                benchmark_annual = (1 + benchmark_total) ** (
                    1 / (len(benchmark_returns) / annualization_factor)
                ) - 1
                info_ratio = (
                    (annual_return - benchmark_annual) / tracking_error if tracking_error > 0 else 0
                )
                metrics["benchmark_return"] = benchmark_total
                metrics["excess_return"] = total_return - benchmark_total
                metrics["information_ratio"] = info_ratio
        return metrics

    def calculate_rolling_ic_stats(
        self, signals: pd.Series, forward_returns: pd.Series, window: int = 21
    ) -> Dict[str, float]:
        aligned_signals = signals.shift(1)
        rolling_ic = aligned_signals.rolling(window).corr(forward_returns)
        valid_ic = rolling_ic.dropna()
        if len(valid_ic) < 10:
            return {"mean_ic": 0.0, "std_ic": 0.0, "pct_positive": 0.0, "ic_ir": 0.0}
        mean_ic = valid_ic.mean()
        std_ic = valid_ic.std()
        pct_positive = (valid_ic > 0).mean()
        ic_ir = mean_ic / std_ic if std_ic > 0 else 0.0
        return {"mean_ic": mean_ic, "std_ic": std_ic, "pct_positive": pct_positive, "ic_ir": ic_ir}

    def calculate_deflated_sharpe_ratio(
        self,
        sharpe: float,
        num_observations: int,
        num_trials: int,
        skewness: float = 0,
        kurtosis: float = 3,
    ) -> float:
        if num_trials <= 1:
            return self.calculate_probabilistic_sharpe_ratio(
                sharpe, num_observations, 0, skewness, kurtosis
            )
        try:
            euler = 0.5772156649
            z_arg = max(0.0001, min(0.9999, 1 - 1 / num_trials))
            e_max_arg = max(0.0001, min(0.9999, 1 - 1 / (num_trials * np.e)))
            expected_max_sr = (1 - euler) * stats.norm.ppf(z_arg) + euler * stats.norm.ppf(
                e_max_arg
            )
            sr_std = np.sqrt(
                (1 + 0.5 * sharpe**2 - skewness * sharpe + (kurtosis - 3) / 4 * sharpe**2)
                / num_observations
            )
            expected_max_sr *= sr_std
            z_score = (sharpe - expected_max_sr) / sr_std if sr_std > 0 else 0
            return max(0, min(1, stats.norm.cdf(z_score)))
        except:
            return 0.5

    def calculate_probabilistic_sharpe_ratio(
        self,
        sharpe: float,
        num_observations: int,
        benchmark_sharpe: float = 0,
        skewness: float = 0,
        kurtosis: float = 3,
    ) -> float:
        if num_observations < 2:
            return 0.5
        variance_term = max(
            0.01, 1 + 0.5 * sharpe**2 - skewness * sharpe + (kurtosis - 3) / 4 * sharpe**2
        )
        sharpe_std = np.sqrt(variance_term / num_observations)
        z_score = (sharpe - benchmark_sharpe) / sharpe_std if sharpe_std > 0 else 0
        return max(0, min(1, stats.norm.cdf(z_score)))

    def calculate_hurst_exponent(self, returns: pd.Series, max_lag: int = 100) -> float:
        returns = returns.dropna().values
        n = len(returns)
        if n < 50:
            return 0.5
        max_lag = min(max_lag, n // 4)
        rs_values = []
        for lag in range(10, max_lag):
            num_chunks = n // lag
            rs_list = []
            for i in range(num_chunks):
                chunk = returns[i * lag : (i + 1) * lag]
                mean_c = chunk.mean()
                cum_dev = np.cumsum(chunk - mean_c)
                r = cum_dev.max() - cum_dev.min()
                s = chunk.std()
                if s > 0:
                    rs_list.append(r / s)
            if rs_list:
                rs_values.append((np.log(lag), np.log(np.mean(rs_list))))
        if len(rs_values) < 2:
            return 0.5
        rs_arr = np.array(rs_values)
        slope, _, _, _, _ = stats.linregress(rs_arr[:, 0], rs_arr[:, 1])
        return max(0, min(1, slope))

    def calculate_cdar(self, returns: pd.Series, alpha: float = 0.05) -> float:
        cumulative = (1 + returns).cumprod()
        running_max = cumulative.cummax()
        drawdowns = (cumulative - running_max) / running_max
        threshold = drawdowns.quantile(alpha)
        worst = drawdowns[drawdowns <= threshold]
        return worst.mean() if len(worst) > 0 else drawdowns.min()

    def calculate_parameter_stability_index(self, sharpes: List[float]) -> float:
        valid = [s for s in sharpes if not np.isnan(s) and s != 0]
        if len(valid) < 2:
            return 0.5
        mean_s = np.mean(valid)
        std_s = np.std(valid)
        if mean_s != 0:
            return 1 / (1 + abs(std_s / mean_s))
        return 0.5 if std_s == 0 else 0.3

    def calculate_global_robustness_index(self, metrics: Dict[str, float]) -> float:
        weights = {"dsr": 0.2, "psr": 0.2, "wfe": 0.2, "hurst": 0.1, "cdar": 0.15, "psi": 0.15}
        normalized = {
            "dsr": max(0, min(1, metrics.get("dsr", 0.5))),
            "psr": max(0, min(1, metrics.get("psr", 0.5))),
            "wfe": max(0, min(1, metrics.get("wfe", 0.5) / 1.5)),
            "hurst": max(0, min(1, metrics.get("hurst", 0.5))),
            "cdar": max(0, min(1, 1 + metrics.get("cdar", -0.2) * 2)),
            "psi": max(0, min(1, metrics.get("psi", 0.5))),
        }
        return sum((weights[k] * normalized[k] for k in weights))


class HyperparameterOptimizer:

    def __init__(self, config: StrategyConfig, benchmark_returns: Optional[pd.Series] = None):
        self.config = config
        self.analytics = PerformanceAnalytics(config)
        self.benchmark_returns = benchmark_returns

    def generate_param_grid(self) -> List[Dict[str, Any]]:
        if self.config.fast_mode:
            vol_lb = [20]
            q_win = [252]
            r_q = [0.5]
            pers = [3]
            cons_mode = ["moderate"]
            min_votes = [2]
            zs_short = [10]
            zs_med = [20]
            zs_long = [40]
            multitf_mode = ["majority"]
            rsi_per = [14]
            rsi_os = [30]
            rsi_ob = [70]
            exit_mode = ["any"]
        else:
            vol_lb = self.config.vol_lookback_grid
            q_win = self.config.quantile_window_grid
            r_q = self.config.regime_quantile_grid
            pers = self.config.persistence_days_grid
            cons_mode = self.config.consensus_mode_grid
            min_votes = self.config.min_votes_grid
            zs_short = self.config.zscore_short_lookback_grid
            zs_med = self.config.zscore_medium_lookback_grid
            zs_long = self.config.zscore_long_lookback_grid
            multitf_mode = self.config.multitf_consensus_mode_grid
            rsi_per = self.config.rsi_period_grid
            rsi_os = self.config.rsi_oversold_grid
            rsi_ob = self.config.rsi_overbought_grid
            exit_mode = self.config.exit_mode_grid
        params = []
        for vl, qw, rq, pd, cm, mv, zss, zsm, zsl, mtf, rp, ros, rob, em in product(
            vol_lb,
            q_win,
            r_q,
            pers,
            cons_mode,
            min_votes,
            zs_short,
            zs_med,
            zs_long,
            multitf_mode,
            rsi_per,
            rsi_os,
            rsi_ob,
            exit_mode,
        ):
            if qw >= vl and ros < rob and (zss < zsm < zsl):
                params.append(
                    {
                        "vol_lookback": vl,
                        "quantile_window": qw,
                        "regime_quantile": rq,
                        "persistence_days": pd,
                        "consensus_mode": cm,
                        "min_votes": mv,
                        "zscore_short_lookback": zss,
                        "zscore_medium_lookback": zsm,
                        "zscore_long_lookback": zsl,
                        "multitf_consensus_mode": mtf,
                        "rsi_period": rp,
                        "rsi_oversold": ros,
                        "rsi_overbought": rob,
                        "exit_mode": em,
                    }
                )
        if len(params) > self.config.max_param_combinations:
            np.random.seed(42)
            idx = np.random.choice(len(params), self.config.max_param_combinations, replace=False)
            params = [params[i] for i in idx]
        return params

    def evaluate_params(self, params: Dict, df: pd.DataFrame) -> Dict[str, float]:
        try:
            detector = ThresholdRegimeDetectorWithConsensus(
                self.config,
                vol_lookback=params["vol_lookback"],
                quantile_window=params["quantile_window"],
                regime_quantile=params["regime_quantile"],
                persistence_days=params["persistence_days"],
                consensus_mode=params["consensus_mode"],
                min_votes=params["min_votes"],
                zscore_short_lookback=params["zscore_short_lookback"],
                zscore_medium_lookback=params["zscore_medium_lookback"],
                zscore_long_lookback=params["zscore_long_lookback"],
                multitf_consensus_mode=params["multitf_consensus_mode"],
                rsi_period=params["rsi_period"],
                rsi_oversold=params["rsi_oversold"],
                rsi_overbought=params["rsi_overbought"],
                exit_mode=params["exit_mode"],
            )
            signals, _, _ = detector.generate_signals_with_consensus(df)
            if signals.dropna().shape[0] < 60:
                return {"sharpe_ratio": -999, "valid": False}
            df_regimes = detector.detect_regimes(df)
            engine = BacktestEngine(self.config, self.benchmark_returns)
            results = engine.run_backtest(df_regimes, signals)
            returns = results["strategy_return"].dropna()
            if len(returns) < 30:
                return {"sharpe_ratio": -999, "valid": False}
            metrics = self.analytics.calculate_basic_metrics(returns, self.benchmark_returns)
            metrics["valid"] = True
            metrics["num_trades"] = results["is_execution_day"].sum()
            return metrics
        except Exception as e:
            return {"sharpe_ratio": -999, "valid": False, "error": str(e)}

    def optimize(self, df: pd.DataFrame, verbose: bool = False) -> Tuple[Dict, Dict]:
        grid = self.generate_param_grid()
        if verbose:
            print(f"    Testing {len(grid)} parameter combinations (multi-TF Z-score enabled)...")
        best_params, best_obj, best_metrics = (None, -np.inf, {})
        for params in grid:
            metrics = self.evaluate_params(params, df)
            if not metrics.get("valid"):
                continue
            obj = metrics.get("sharpe_ratio", -999)
            if obj > best_obj:
                best_obj, best_params, best_metrics = (obj, params, metrics)
        if best_params is None:
            best_params = {
                "vol_lookback": self.config.vol_lookback,
                "quantile_window": self.config.quantile_window,
                "regime_quantile": self.config.regime_quantile,
                "persistence_days": self.config.persistence_days,
                "consensus_mode": "moderate",
                "min_votes": 2,
                "zscore_short_lookback": 10,
                "zscore_medium_lookback": 20,
                "zscore_long_lookback": 40,
                "multitf_consensus_mode": "majority",
                "rsi_period": 14,
                "rsi_oversold": 30,
                "rsi_overbought": 70,
                "exit_mode": "any",
            }
        if verbose:
            print(
                f"    Best: multi-TF=[{best_params['zscore_short_lookback']},{best_params['zscore_medium_lookback']},{best_params['zscore_long_lookback']}], mode={best_params['multitf_consensus_mode']}, Sharpe={best_obj:.3f}"
            )
        return (best_params, best_metrics)


class WalkForwardValidator:

    def __init__(self, config: StrategyConfig, benchmark_returns: Optional[pd.Series] = None):
        self.config = config
        self.analytics = PerformanceAnalytics(config)
        self.optimizer = HyperparameterOptimizer(config, benchmark_returns)
        self.benchmark_returns = benchmark_returns

    def run_walk_forward(self, df: pd.DataFrame, verbose: bool = True) -> Dict[str, Any]:
        detector = ThresholdRegimeDetectorWithConsensus(self.config)
        df_init = detector.detect_regimes(df)
        max_warmup = max(
            max(self.config.quantile_window_grid) + max(self.config.vol_lookback_grid),
            max(self.config.zscore_long_lookback_grid),
        )
        valid_idx = df_init.dropna(subset=["regime", "vol_threshold"]).index
        min_req = self.config.min_train_window + self.config.validation_window + max_warmup
        if len(valid_idx) < min_req:
            raise ValueError(f"Insufficient data: need {min_req}, have {len(valid_idx)}")
        fold_results, oos_returns_list, param_history = ([], [], [])
        num_trials = len(self.optimizer.generate_param_grid())
        fold_num, train_end_idx = (0, self.config.min_train_window)
        carry_forward_capital = self.config.initial_capital
        if verbose:
            print(f"\n{'=' * 80}\nWALK-FORWARD VALIDATION - V2.5.2 (MULTI-TIMEFRAME Z-SCORE)")
            print(
                f"Mode: {('Continuous Equity' if self.config.continuous_equity else 'Independent Folds')}"
            )
            print(
                f"Multi-TF Z-Score: [{self.config.zscore_short_lookback}, {self.config.zscore_medium_lookback}, {self.config.zscore_long_lookback}]"
            )
            print(f"Multi-TF Consensus: {self.config.multitf_consensus_mode.upper()}")
            print(
                f"Percentile Thresholds: {('ENABLED' if self.config.use_percentile_thresholds else 'DISABLED')}"
            )
            print(f"{'=' * 80}")
        while train_end_idx + self.config.validation_window <= len(valid_idx):
            fold_num += 1
            train_idx = valid_idx[:train_end_idx]
            val_idx = valid_idx[train_end_idx : train_end_idx + self.config.validation_window]
            if verbose:
                print(
                    f"\nFold {fold_num}: Train {train_idx[0].date()}-{train_idx[-1].date()}, Val {val_idx[0].date()}-{val_idx[-1].date()}"
                )
            train_df = df.loc[train_idx]
            opt_params, is_metrics = self.optimizer.optimize(train_df, verbose=verbose)
            param_history.append({"fold": fold_num, **opt_params})
            is_sharpe = is_metrics.get("sharpe_ratio", 0)
            opt_detector = ThresholdRegimeDetectorWithConsensus(
                self.config, **{k: v for k, v in opt_params.items() if k != "fold"}
            )
            full_val_df = df.loc[: val_idx[-1]]
            val_signals, val_diagnostics, val_consensus_diag = (
                opt_detector.generate_signals_with_consensus(full_val_df)
            )
            val_signals = val_signals.loc[val_idx]
            df_regimes_val = opt_detector.detect_regimes(full_val_df).loc[val_idx]
            if self.config.continuous_equity:
                fold_config = StrategyConfig()
                fold_config.initial_capital = carry_forward_capital
                engine = BacktestEngine(fold_config, self.benchmark_returns)
            else:
                engine = BacktestEngine(self.config, self.benchmark_returns)
            val_results = engine.run_backtest(df_regimes_val, val_signals)
            val_returns = val_results["strategy_return"].dropna()
            oos_returns_list.append(val_returns)
            if self.config.continuous_equity:
                carry_forward_capital = val_results["portfolio_value"].iloc[-1]
            oos_metrics = (
                self.analytics.calculate_basic_metrics(val_returns, self.benchmark_returns)
                if len(val_returns) > 10
                else {}
            )
            oos_sharpe = oos_metrics.get("sharpe_ratio", 0)
            wfe = oos_sharpe / is_sharpe if is_sharpe != 0 else 0
            forward_returns = (
                df_regimes_val["returns"]
                if "returns" in df_regimes_val.columns
                else df_regimes_val["Close"].pct_change()
            )
            ic_stats = self.analytics.calculate_rolling_ic_stats(
                val_signals, forward_returns, window=21
            )
            multitf_3votes = (
                val_consensus_diag["multitf_oversold_votes"]
                + val_consensus_diag["multitf_overbought_votes"]
                == 3
            ).sum()
            multitf_2votes = (
                val_consensus_diag["multitf_oversold_votes"]
                + val_consensus_diag["multitf_overbought_votes"]
                == 2
            ).sum()
            fold_results.append(
                {
                    "fold": fold_num,
                    "is_sharpe": is_sharpe,
                    "oos_sharpe": oos_sharpe,
                    "oos_return": oos_metrics.get("total_return", 0),
                    "wfe": wfe,
                    "mean_ic": ic_stats["mean_ic"],
                    "std_ic": ic_stats["std_ic"],
                    "pct_ic_positive": ic_stats["pct_positive"],
                    "ic_ir": ic_stats["ic_ir"],
                    "oos_long_days": (val_results["position_type"] == "LONG").sum(),
                    "oos_short_days": (val_results["position_type"] == "SHORT").sum(),
                    "oos_flat_days": (val_results["position_type"] == "FLAT").sum(),
                    "multitf_3votes": multitf_3votes,
                    "multitf_2votes": multitf_2votes,
                    "ending_capital": (
                        val_results["portfolio_value"].iloc[-1]
                        if self.config.continuous_equity
                        else self.config.initial_capital
                    ),
                    **{f"param_{k}": v for k, v in opt_params.items()},
                }
            )
            if verbose:
                print(f"  IS Sharpe: {is_sharpe:.3f}, OOS Sharpe: {oos_sharpe:.3f}, WFE: {wfe:.3f}")
                print(f"  Rolling IC: mean={ic_stats['mean_ic']:.3f}, IR={ic_stats['ic_ir']:.3f}")
                print(f"  Multi-TF Consensus: 3/3={multitf_3votes}, 2/3={multitf_2votes}")
            train_end_idx += self.config.step_size
        all_oos = pd.concat(oos_returns_list)
        agg = self.analytics.calculate_basic_metrics(all_oos, self.benchmark_returns)
        valid_wfes = [
            f["wfe"] for f in fold_results if not np.isnan(f["wfe"]) and abs(f["wfe"]) < 10
        ]
        avg_wfe = np.mean(valid_wfes) if valid_wfes else 0
        avg_ic = (
            np.mean([f["mean_ic"] for f in fold_results if not np.isnan(f["mean_ic"])])
            if fold_results
            else 0
        )
        avg_ic_ir = (
            np.mean([f["ic_ir"] for f in fold_results if not np.isnan(f["ic_ir"])])
            if fold_results
            else 0
        )
        skew, kurt = (all_oos.skew(), all_oos.kurtosis() + 3)
        dsr = self.analytics.calculate_deflated_sharpe_ratio(
            agg.get("sharpe_ratio", 0), len(all_oos), num_trials * len(fold_results), skew, kurt
        )
        psr = self.analytics.calculate_probabilistic_sharpe_ratio(
            agg.get("sharpe_ratio", 0), len(all_oos), self.config.psr_benchmark_sharpe, skew, kurt
        )
        hurst = self.analytics.calculate_hurst_exponent(all_oos)
        cdar = self.analytics.calculate_cdar(all_oos)
        psi = self.analytics.calculate_parameter_stability_index(
            [f["oos_sharpe"] for f in fold_results]
        )
        gri = self.analytics.calculate_global_robustness_index(
            {"dsr": dsr, "psr": psr, "wfe": avg_wfe, "hurst": hurst, "cdar": cdar, "psi": psi}
        )
        is_robust = (
            gri > self.config.gri_threshold
            and psr >= self.config.min_psr
            and (avg_wfe >= self.config.min_wfe)
        )
        aggregate = {
            "num_folds": len(fold_results),
            "oos_total_return": agg.get("total_return", 0),
            "oos_sharpe": agg.get("sharpe_ratio", 0),
            "oos_max_drawdown": agg.get("max_drawdown", 0),
            "avg_wfe": avg_wfe,
            "avg_ic": avg_ic,
            "avg_ic_ir": avg_ic_ir,
            "dsr": dsr,
            "psr": psr,
            "hurst": hurst,
            "cdar": cdar,
            "psi": psi,
            "gri": gri,
            "is_robust": is_robust,
            "total_long_days": sum((f["oos_long_days"] for f in fold_results)),
            "total_short_days": sum((f["oos_short_days"] for f in fold_results)),
            "total_flat_days": sum((f["oos_flat_days"] for f in fold_results)),
            "total_multitf_3votes": sum((f["multitf_3votes"] for f in fold_results)),
            "total_multitf_2votes": sum((f["multitf_2votes"] for f in fold_results)),
            "final_capital": carry_forward_capital if self.config.continuous_equity else None,
        }
        if verbose:
            print(
                f"\n{'-' * 80}\nAGGREGATE: OOS Sharpe={agg.get('sharpe_ratio', 0):.3f}, GRI={gri:.3f}, PSR={psr:.3f}, WFE={avg_wfe:.3f}"
            )
            print(f"Rolling IC: avg={avg_ic:.3f}, IC IR={avg_ic_ir:.3f}")
            print(
                f"Multi-TF Consensus Quality: 3/3={aggregate['total_multitf_3votes']}, 2/3={aggregate['total_multitf_2votes']}"
            )
            if self.config.continuous_equity:
                print(
                    f"Continuous Equity: ${self.config.initial_capital:,.0f} -> ${carry_forward_capital:,.0f}"
                )
            print(f"DEPLOYMENT: {('READY' if is_robust else 'NOT READY')}")
        return {
            "fold_results": fold_results,
            "aggregate_metrics": aggregate,
            "oos_returns": all_oos,
            "param_history": param_history,
        }


def main():
    print("=" * 80)
    print(f"THRESHOLD REGIME STRATEGY V2.5.2 - {TICKER}")
    print("ENHANCEMENT: Multi-Timeframe Z-Score Consensus + Percentile Thresholds")
    print("=" * 80)
    config = StrategyConfig()
    print(f"\n[CONFIG] {config.ticker}, {config.start_date} to {config.end_date}")
    print(f"  Capital: ${config.initial_capital:,.0f}, Leverage: {config.leverage}x")
    print(f"  Mode: {('Production' if config.compounding_mode else 'Research')}")
    print(f"\n[MULTI-TF Z-SCORE]")
    print(
        f"  Lookbacks: Short={config.zscore_short_lookback}, Medium={config.zscore_medium_lookback}, Long={config.zscore_long_lookback}"
    )
    print(f"  Consensus Mode: {config.multitf_consensus_mode.upper()}")
    print(
        f"  Percentile Thresholds: {('ENABLED' if config.use_percentile_thresholds else 'DISABLED')}"
    )
    print(f"  Multi-TF Weight: {config.multitf_zscore_weight}x")
    print("\n[DATA] Fetching from Yahoo Finance...")
    start_dt = datetime.strptime(config.start_date, "%Y-%m-%d")
    ext_start = (start_dt - timedelta(days=config.warmup_buffer_days)).strftime("%Y-%m-%d")
    fetcher = YahooDataFetcher()
    try:
        df_full = fetcher.fetch_historical_data(config.ticker, ext_start, config.end_date)
    except ValueError as e:
        print(f"[ERROR] {e}")
        return None
    try:
        benchmark_df = fetcher.fetch_historical_data(
            config.benchmark_ticker, ext_start, config.end_date
        )
        benchmark_returns = benchmark_df["Close"].pct_change()
        print(f"  Benchmark {config.benchmark_ticker}: {len(benchmark_df)} days")
    except:
        print(f"  [WARNING] Could not fetch {config.benchmark_ticker}")
        benchmark_returns = None
    print("\n[WALK-FORWARD VALIDATION]")
    wf = WalkForwardValidator(config, benchmark_returns)
    wf_results = wf.run_walk_forward(df_full, verbose=True)
    print("\n[FULL BACKTEST] Filtering to backtest period...")
    final_params = wf_results["param_history"][-1] if wf_results["param_history"] else {}
    final_params = {k: v for k, v in final_params.items() if k != "fold"}
    detector = ThresholdRegimeDetectorWithConsensus(config, **final_params)
    df_regimes_full = detector.detect_regimes(df_full)
    signals_full, diagnostics_full, consensus_diag_full = detector.generate_signals_with_consensus(
        df_full
    )
    backtest_start = pd.to_datetime(config.start_date)
    backtest_end = pd.to_datetime(config.end_date)
    df_backtest = df_regimes_full[
        (df_regimes_full.index >= backtest_start) & (df_regimes_full.index <= backtest_end)
    ]
    signals_backtest = signals_full[
        (signals_full.index >= backtest_start) & (signals_full.index <= backtest_end)
    ]
    diagnostics_backtest = diagnostics_full[
        (diagnostics_full.index >= backtest_start) & (diagnostics_full.index <= backtest_end)
    ]
    consensus_diag_backtest = consensus_diag_full[
        (consensus_diag_full.index >= backtest_start) & (consensus_diag_full.index <= backtest_end)
    ]
    if benchmark_returns is not None:
        benchmark_backtest = benchmark_returns[
            (benchmark_returns.index >= backtest_start) & (benchmark_returns.index <= backtest_end)
        ]
    else:
        benchmark_backtest = None
    print(f"  Backtest period: {df_backtest.index[0].date()} to {df_backtest.index[-1].date()}")
    print(f"  Trading days: {len(df_backtest)}")
    engine = BacktestEngine(config, benchmark_backtest)
    bt_results = engine.run_backtest(df_backtest, signals_backtest)
    final_value = bt_results["portfolio_value"].iloc[-1]
    total_return = (final_value / config.initial_capital - 1) * 100
    print(f"\n  Initial Capital: ${config.initial_capital:,.2f}")
    print(f"  Final Value: ${final_value:,.2f}")
    print(f"  Total Return: {total_return:.2f}%")
    print(f"  Trades: {bt_results['is_execution_day'].sum()}")
    print(
        f"  LONG: {(bt_results['position_type'] == 'LONG').sum()}, SHORT: {(bt_results['position_type'] == 'SHORT').sum()}, FLAT: {(bt_results['position_type'] == 'FLAT').sum()}"
    )
    total_multitf_3votes = (
        consensus_diag_backtest["multitf_oversold_votes"]
        + consensus_diag_backtest["multitf_overbought_votes"]
        == 3
    ).sum()
    total_multitf_2votes = (
        consensus_diag_backtest["multitf_oversold_votes"]
        + consensus_diag_backtest["multitf_overbought_votes"]
        == 2
    ).sum()
    print(f"\n[MULTI-TF CONSENSUS QUALITY]")
    print(f"  3/3 timeframes agree: {total_multitf_3votes} days")
    print(f"  2/3 timeframes agree: {total_multitf_2votes} days")
    print("\n[EXPORT]")
    out = f"results/consensus-mean-reversion-momentum/{TICKER.lower()}_v252_multitf_zscore"
    bt_results.to_csv(f"{out}_backtest.csv")
    pd.DataFrame(wf_results["fold_results"]).to_csv(f"{out}_walkforward.csv", index=False)
    pd.DataFrame(wf_results["param_history"]).to_csv(f"{out}_params.csv", index=False)
    diagnostics_backtest.to_csv(f"{out}_diagnostics.csv")
    consensus_diag_backtest.to_csv(f"{out}_consensus_diagnostics.csv")
    with open(f"{out}_metrics.json", "w") as f:
        json.dump(wf_results["aggregate_metrics"], f, indent=2, default=str)
    print(f"  Saved to: {out}_*.csv/json")
    agg = wf_results["aggregate_metrics"]
    print(f"\n{'=' * 80}\nSUMMARY V2.5.2:")
    print(f"OOS Sharpe: {agg['oos_sharpe']:.3f}, GRI: {agg['gri']:.3f}")
    print(
        f"Multi-TF Z-Score Consensus: 3/3={agg['total_multitf_3votes']}, 2/3={agg['total_multitf_2votes']}"
    )
    print(f"DEPLOYMENT: {('READY' if agg['is_robust'] else 'NOT READY')}\n{'=' * 80}")
    return {
        "config": config,
        "backtest": bt_results,
        "wf": wf_results,
        "diagnostics": diagnostics_backtest,
        "consensus_diagnostics": consensus_diag_backtest,
    }


if __name__ == "__main__":
    results = main()
